# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：411303106　姓名：林苡琦

## 1. 系統目的（一句話）
讓民眾用一句話完成診所內科、外科、牙科的掛號（取得號碼）、取消（歸還號碼）與名額查詢；系統不提供醫療建議，也不推薦科別。

## 2. 使用者會怎麼說（至少 5 句）
1. 我要掛外科
2. 幫我取消外科掛號
3. 外科現在還有幾個名額？
4. 麻煩幫我登記今天的外科門診
5. 牙科的掛號我不要了
6. 我肚子好痛，應該看哪一科？（不可推薦科別）

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| departments | id | INTEGER | 主鍵，自動編號 |
| departments | name | TEXT UNIQUE | 科別名稱 |
| departments | quota | INTEGER | 每天總名額 |
| departments | remaining | INTEGER | 剩餘名額 |
| appointments | id | INTEGER | 主鍵，自動編號 |
| appointments | patient | TEXT | 病患姓名 |
| appointments | department | TEXT | 掛號科別 |
| appointments | number | INTEGER | 掛號號碼（每科從 1 號開始） |
| appointments | status | TEXT | 已掛號／已取消 |

初始資料：
- 內科：名額 5
- 外科：名額 3
- 牙科：名額 1（方便測試額滿）
- appointments 一開始是空的

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：掛號、取消、查詢
- 對象只能是：內科、外科、牙科
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：提供醫療建議、根據症狀推薦科別、回覆多餘文字。
- AI 只回「動作,科別」，畫面上的回覆文字都由程式產生，所以不會出現 AI 推薦的內容。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 有姓名、還有名額、沒掛過該科 | 掛號成功，告訴他號碼和剩餘名額 | 剩餘名額 −1；新增一筆「已掛號」紀錄（含號碼） |
| 掛號 | 名額已滿 | 名額已滿 | 無 |
| 掛號 | 已掛過該科 | 不能重複掛號 | 無 |
| 取消 | 有姓名、找得到掛號紀錄 | 取消成功，告訴他號碼已歸還 | 紀錄改為「已取消」（號碼作廢）；剩餘名額 +1 |
| 取消 | 找不到紀錄 | 查無掛號紀錄 | 無 |
| 查詢 | 科別存在 | 顯示總名額與剩餘名額 | 無 |

**號碼規則：** 每科從 1 號開始，依掛號的先後順序發號（1、2、3…），後掛的人號碼一定比較大。
取消時歸還號碼，那個號碼就作廢，不會再發給別人，所以不會有人插隊。
例：內科 1、2 號已經有人，1 號取消後，下一位掛內科的人拿到 3 號。

## 6. 例外情況（防呆）
1. 輸入空白：請使用者輸入需求。
2. 掛號或取消沒填姓名：請使用者輸入姓名。
3. 呼叫 AI 失敗：當作未知，程式不當掉。
4. AI 回覆多了符號或文字：程式只從回覆中找允許的值。
5. 只說症狀、沒說科別、或說了沒有的科別（如眼科）：不推薦，請他直接說科別或洽詢櫃台。
6. AI 自己猜科別（使用者原話裡沒有該科別）：當作未知。
7. 名額已滿、重複掛號、取消時查無紀錄：拒絕並說明原因。

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛內科 | 掛號,內科 |
| 2 | 正常 | 幫我取消牙科的掛號 | 取消,牙科 |
| 3 | 正常 | 外科現在還有幾個名額？ | 查詢,外科 |
| 4 | 正常 | 麻煩幫我登記今天的外科門診 | 掛號,外科 |
| 5 | 錯誤 | 我想掛眼科 | 科別：未知 |
| 6 | 錯誤 | 今天天氣如何 | 未知,未知 |
| 7 | 錯誤（安全） | 我肚子好痛，應該看哪一科？ | 科別：未知 |
| 8 | 錯誤（安全） | 我牙齒好痛 | 科別：未知 |
| 9 | 邊界 | 牙科的掛號我不要了 | 取消,牙科 |

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [ ]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [ ]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 每次執行都先刪掉舊的表，再重建並放入初始資料
# ============================================================
DB = "clinic.db"
DEPTS = ["內科", "外科", "牙科"]   # 系統只認得這三科

conn = sqlite3.connect(DB)
try:
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS departments")
    cursor.execute("DROP TABLE IF EXISTS appointments")

    # 表 1：科別（總名額、剩餘名額）
    cursor.execute("""
        CREATE TABLE departments (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            name      TEXT UNIQUE NOT NULL,
            quota     INTEGER NOT NULL,
            remaining INTEGER NOT NULL
        )
    """)

    # 表 2：掛號紀錄（number 是掛號號碼；status 是「已掛號」或「已取消」）
    cursor.execute("""
        CREATE TABLE appointments (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            patient    TEXT NOT NULL,
            department TEXT NOT NULL,
            number     INTEGER NOT NULL,
            status     TEXT NOT NULL
        )
    """)

    # 初始資料：(科別, 總名額, 剩餘名額)
    cursor.executemany(
        "INSERT INTO departments (name, quota, remaining) VALUES (?, ?, ?)",
        [
            ("內科", 5, 5),
            ("外科", 3, 3),
            ("牙科", 1, 1),   # 只有 1 個名額，方便測試「額滿」
        ]
    )
    conn.commit()
finally:
    conn.close()   # 不管有沒有出錯，一定關閉連線

print("✅ 資料庫建立完成")

✅ 資料庫建立完成


In [ ]:
# ============================================================
# 第 3 格：顯示資料表（確認資料庫的內容）
# ============================================================
def show_tables():
    conn = sqlite3.connect(DB)
    try:
        print("🏥 departments（科別）")
        display(pd.read_sql_query("SELECT * FROM departments", conn))
        print("📝 appointments（掛號紀錄）")
        display(pd.read_sql_query("SELECT * FROM appointments", conn))
    finally:
        conn.close()

show_tables()

🏥 departments（科別）


,id,name,quota,remaining
0,1,內科,5,5
1,2,外科,3,3
2,3,牙科,1,1


📝 appointments（掛號紀錄）


,id,patient,department,number,status


In [ ]:
# ============================================================
# 第 4 格：AI 解讀使用者的話 → 回傳 (動作, 科別, AI 原始回覆)
# ============================================================
# 注意：「取消」要排在「掛號」前面。
# 因為「取消掛號」裡也有「掛號」兩個字，先找「取消」才不會判斷錯。
ACTIONS = ["取消", "查詢", "掛號"]

def ai_parse(user_say):
    prompt = f"""你是診所的掛號助理，只判斷使用者想做什麼。
科別只有：內科、外科、牙科。

只能回覆一行，格式為：動作,科別
- 動作只能是：掛號、取消、查詢、未知
- 科別只能是：內科、外科、牙科、未知
- 使用者沒有親口說出科別名稱時，科別填 未知（不可以根據症狀猜科別）
- 不要回覆任何其他文字

範例：
使用者：我要掛內科 → 掛號,內科
使用者：幫我取消外科的掛號 → 取消,外科
使用者：牙科還有名額嗎 → 查詢,牙科
使用者：我牙齒很痛 → 未知,未知
使用者：今天天氣如何 → 未知,未知

使用者：{user_say}"""

    # 步驟 1：呼叫 AI
    # 防呆：網路或金鑰出問題時，不讓程式當掉，直接回傳未知
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print("⚠️ 呼叫 AI 失敗：", e)
        return "未知", "未知", ""

    # 步驟 2：從 AI 的回覆中「找」出動作
    # 防呆：AI 多回了符號或文字也沒關係，只要找得到就好
    action = "未知"
    for a in ACTIONS:
        if a in raw:
            action = a
            break

    # 步驟 3：從 AI 的回覆中「找」出科別
    dept = "未知"
    for d in DEPTS:
        if d in raw:
            dept = d
            break

    # 步驟 4：防呆：使用者自己沒說出這個科別，代表是 AI 猜的 → 當作未知
    if dept not in user_say:
        dept = "未知"

    return action, dept, raw

In [ ]:
# ============================================================
# 第 5 格：測試清單（和規格書第 7 節一樣）
# 格式：(使用者說的話, 預期動作, 預期科別)，"*" 表示不檢查這一項
# ============================================================
test_list = [
    ("我要掛內科",                   "掛號", "內科"),   # 正常
    ("幫我取消牙科的掛號",           "取消", "牙科"),   # 正常
    ("外科現在還有幾個名額？",       "查詢", "外科"),   # 正常
    ("麻煩幫我登記今天的外科門診",   "掛號", "外科"),   # 正常：說法不一樣
    ("我想掛眼科",                   "*",    "未知"),   # 錯誤：沒有這科
    ("今天天氣如何",                 "未知", "未知"),   # 錯誤：無關的話
    ("我肚子好痛，應該看哪一科？",   "*",    "未知"),   # 安全：不可推薦科別
    ("我牙齒好痛",                   "*",    "未知"),   # 安全：不可自己猜牙科
    ("牙科的掛號我不要了",           "取消", "牙科"),   # 邊界：句子裡也有「掛號」
]

passed = 0
for say, exp_action, exp_dept in test_list:
    action, dept, raw = ai_parse(say)
    ok = (exp_action == "*" or exp_action == action) and (exp_dept == "*" or exp_dept == dept)
    if ok:
        passed += 1
    print("✅" if ok else "❌", say)
    print("     AI 原始回覆：", repr(raw))
    print(f"     系統解讀：{action} / {dept}（預期：{exp_action} / {exp_dept}）")
    print()

print(f"通過 {passed} / {len(test_list)} 句")

✅ 我要掛內科
     AI 原始回覆： '掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

✅ 幫我取消牙科的掛號
     AI 原始回覆： '取消,牙科'
     系統解讀：取消 / 牙科（預期：取消 / 牙科）

✅ 外科現在還有幾個名額？
     AI 原始回覆： '查詢,外科'
     系統解讀：查詢 / 外科（預期：查詢 / 外科）

✅ 麻煩幫我登記今天的外科門診
     AI 原始回覆： '掛號,外科'
     系統解讀：掛號 / 外科（預期：掛號 / 外科）

✅ 我想掛眼科
     AI 原始回覆： '掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 今天天氣如何
     AI 原始回覆： '未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 我肚子好痛，應該看哪一科？
     AI 原始回覆： '查詢,未知'
     系統解讀：查詢 / 未知（預期：* / 未知）

✅ 我牙齒好痛
     AI 原始回覆： '未知,未知'
     系統解讀：未知 / 未知（預期：* / 未知）

✅ 牙科的掛號我不要了
     AI 原始回覆： '取消,牙科'
     系統解讀：取消 / 牙科（預期：取消 / 牙科）

通過 9 / 9 句


In [ ]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# 每個函式都：fetchone() 後先檢查 None；用 try/finally 關閉連線
# ============================================================
def book(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()

        # 查剩餘名額
        cursor.execute("SELECT remaining FROM departments WHERE name = ?", (dept,))
        row = cursor.fetchone()
        if row is None:
            return f"本診所沒有{dept}。"
        if row[0] <= 0:
            return f"{dept}今天的名額已滿，請改天再掛或洽詢櫃台。"

        # 防呆：同一個人不能重複掛同一科
        cursor.execute(
            "SELECT id FROM appointments WHERE patient = ? AND department = ? AND status = ?",
            (patient, dept, "已掛號")
        )
        if cursor.fetchone() is not None:
            return f"{patient}已經掛過{dept}了，不能重複掛號。"

        # 給號碼：算出這一科「總共發過幾個號碼」（包含已取消的），再 +1
        # 例：發過 1、2 號（不管有沒有取消）→ 下一位拿 3 號
        # 號碼只會往上加，所以後掛的人一定排在先掛的人後面
        cursor.execute("SELECT COUNT(*) FROM appointments WHERE department = ?", (dept,))
        number = cursor.fetchone()[0] + 1   # COUNT(*) 一定有結果（沒有資料時是 0），不會是 None

        # 掛號：名額 -1，新增一筆紀錄（含號碼）
        cursor.execute("UPDATE departments SET remaining = remaining - 1 WHERE name = ?", (dept,))
        cursor.execute(
            "INSERT INTO appointments (patient, department, number, status) VALUES (?, ?, ?, ?)",
            (patient, dept, number, "已掛號")
        )
        conn.commit()
        return f"✅ {patient}掛號{dept}成功！你的號碼是 {number} 號。{dept}還剩 {row[0] - 1} 個名額。"
    finally:
        conn.close()


def cancel(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()

        # 找這個人在這一科的掛號紀錄
        cursor.execute(
            "SELECT id, number FROM appointments WHERE patient = ? AND department = ? AND status = ?",
            (patient, dept, "已掛號")
        )
        row = cursor.fetchone()
        if row is None:
            return f"查無{patient}的{dept}掛號紀錄。"

        # 取消：紀錄改成已取消，名額 +1
        # 狀態改成「已取消」，等於把號碼歸還；這個號碼作廢，不會再發給別人
        cursor.execute("UPDATE appointments SET status = ? WHERE id = ?", ("已取消", row[0]))
        cursor.execute("UPDATE departments SET remaining = remaining + 1 WHERE name = ?", (dept,))
        conn.commit()
        return f"✅ {patient}的{dept}掛號已取消，{row[1]} 號已歸還（此號作廢，不會再發給別人）。"
    finally:
        conn.close()


def query(dept):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT quota, remaining FROM departments WHERE name = ?", (dept,))
        row = cursor.fetchone()
        if row is None:
            return f"本診所沒有{dept}。"
        return f"{dept}今天共 {row[0]} 個名額，還剩 {row[1]} 個。"
    finally:
        conn.close()

In [ ]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# ============================================================
def clinic_system(user_say, patient):
    # 防呆：輸入是空白的，就不呼叫 AI
    if not user_say or not user_say.strip():
        return "請輸入你要掛號、取消或查詢的科別。"
    patient = (patient or "").strip()   # 姓名沒填時當作空字串

    action, dept, raw = ai_parse(user_say)
    print(f"📌 AI 解讀：動作={action}，科別={dept}（原始回覆：{raw!r}）")

    # 安全規定：沒有明確科別 → 不推薦，請他直接說科別
    if dept == "未知":
        return "請直接告訴我要掛哪一科（內科、外科、牙科），或洽詢櫃台。本系統不提供醫療建議。"

    # 查詢不需要姓名
    if action == "查詢":
        return query(dept)

    if action == "未知":
        return f"請說清楚要對{dept}做什麼：掛號、取消，還是查詢名額？"

    # 防呆：掛號和取消一定要有姓名
    if patient == "":
        return "掛號或取消需要姓名，請先輸入你的姓名。"

    if action == "掛號":
        return book(dept, patient)
    return cancel(dept, patient)


# ---------- 完整流程測試（請對照每一行後面的「預期」） ----------
cases = [
    ("我要掛牙科",                   "王小明"),   # 預期：牙科 1 號，牙科剩 0
    ("我也想掛牙科",                 "陳小華"),   # 預期：牙科名額已滿
    ("我要掛內科",                   "王小明"),   # 預期：內科 1 號
    ("我要掛內科",                   "陳小華"),   # 預期：內科 2 號
    ("再幫我掛一次內科",             "王小明"),   # 預期：不能重複掛號
    ("內科的掛號幫我取消",           "王小明"),   # 預期：取消成功，1 號歸還
    ("我要掛內科",                   "林小美"),   # 預期：內科 3 號（排在陳小華後面）
    ("我要取消外科",                 "陳小華"),   # 預期：查無掛號紀錄
    ("我要掛外科",                   ""),         # 預期：請先輸入姓名
    ("",                             "陳小華"),   # 預期：請輸入需求
    ("我頭好痛，該掛哪一科？",       "林小美"),   # 預期：不提供醫療建議
    ("內科還有名額嗎",               "陳小華"),   # 預期：內科共 5 個，剩 3 個
]
for say, name in cases:
    print("-" * 50)
    print(f"🙋 {name or '（未填姓名）'}：{say or '（空白）'}")
    print(clinic_system(say, name))

print("-" * 50)
show_tables()

--------------------------------------------------
🙋 王小明：我要掛牙科
📌 AI 解讀：動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
✅ 王小明掛號牙科成功！你的號碼是 1 號。牙科還剩 0 個名額。
--------------------------------------------------
🙋 陳小華：我也想掛牙科
📌 AI 解讀：動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
牙科今天的名額已滿，請改天再掛或洽詢櫃台。
--------------------------------------------------
🙋 王小明：我要掛內科
📌 AI 解讀：動作=掛號，科別=內科（原始回覆：'掛號,內科'）
✅ 王小明掛號內科成功！你的號碼是 1 號。內科還剩 4 個名額。
--------------------------------------------------
🙋 陳小華：我要掛內科
📌 AI 解讀：動作=掛號，科別=內科（原始回覆：'掛號,內科'）
✅ 陳小華掛號內科成功！你的號碼是 2 號。內科還剩 3 個名額。
--------------------------------------------------
🙋 王小明：再幫我掛一次內科
📌 AI 解讀：動作=掛號，科別=內科（原始回覆：'掛號,內科'）
王小明已經掛過內科了，不能重複掛號。
--------------------------------------------------
🙋 王小明：內科的掛號幫我取消
📌 AI 解讀：動作=取消，科別=內科（原始回覆：'取消,內科'）
✅ 王小明的內科掛號已取消，1 號已歸還（此號作廢，不會再發給別人）。
--------------------------------------------------
🙋 林小美：我要掛內科
📌 AI 解讀：動作=掛號，科別=內科（原始回覆：'掛號,內科'）
✅ 林小美掛號內科成功！你的號碼是 3 號。內科還剩 3 個名額。
--------------------------------------------------
🙋 陳小華：我要取消外科
📌 AI 解讀：動作=取消，科別=外科（原始回覆：'取消,外科'）
查無

,id,name,quota,remaining
0,1,內科,5,3
1,2,外科,3,3
2,3,牙科,1,0


📝 appointments（掛號紀錄）


,id,patient,department,number,status
0,1,王小明,牙科,1,已掛號
1,2,王小明,內科,1,已取消
2,3,陳小華,內科,2,已掛號
3,4,林小美,內科,3,已掛號


In [ ]:
# ============================================================
# 第 8 格：Gradio 網頁（放在最後一格）
# ============================================================
demo = gr.Interface(
    fn=clinic_system,   # 兩個輸入框依序對應 clinic_system(user_say, patient)
    inputs=[
        gr.Textbox(label="💬 你的需求（例如：我要掛內科、取消牙科、外科還有名額嗎）"),
        gr.Textbox(label="🙋 你的姓名（掛號、取消時需要）"),
    ],
    outputs=gr.Textbox(label="🤖 系統回應"),
    title="🏥 診所掛號系統",
    description="科別：內科、外科、牙科。本系統不提供醫療建議。",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0889eaac5cfb772b9a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果